In [275]:
import os

import pandas as pd

In [276]:
topk = 0.05

In [277]:
querynamelist = [
    "Arabidopsis_thaliana",
    # "Erigeron_breviscapus",
    # "Glycine_max",
    # "Zea_mays",
    # "sheetall",
]

In [ ]:
def processing(dfs, subname, topk):
    cachedataframe = dfs.sort_values("score", ascending=False)
    thresholds = dfs["score"].quantile(1 - topk)
    trues = dfs[(dfs["if_right"]) & (dfs["Binding"] == 1)]["enzyme"]
    bigs = cachedataframe[(cachedataframe["score"] > thresholds)]
    rights = bigs[bigs["enzyme"].isin(trues)]
    return len(rights), len(trues)

In [279]:
def pueppcount(result):
    filtered_rows = result[(result["if_right"]) & (result["Binding"] == 1)][
        "substrate"
    ].tolist()
    print("filtered_rows", len(filtered_rows))
    result_grouped = result.groupby("group")
    sum_right_num, sum_all_num, sumsum_num = 0, 0, 0
    for name, group in result_grouped:
        if name in filtered_rows:
            right_num, all_num = processing(group, name, topk)
            sum_right_num += right_num
            sum_all_num += all_num
        sumsum_num += len(group)
        # print("name", len(group), name)
    print("sum_right_num", sum_right_num)
    print("sum_all_num", sum_all_num)
    print("sumsum_num", sumsum_num, "\n")
    return sum_right_num, sum_all_num

In [ ]:
def pueppcombine(queryname, oldpath="../output/", newpath="../others/puepp/"):
    # dfold = pd.read_csv(oldpath + f"{queryname}_ss_nocc.csv")
    dfold = pd.read_csv(oldpath + f"{queryname}_ss.csv")
    dfnew = pd.read_csv(newpath + f"{queryname}_4predict_result.csv")
    df = pd.merge(
        dfold,
        dfnew,
        left_on=["smile", "seq"],
        right_on=["Compound", "Protein"],
        how="inner",
    )
    df.sort_values(by="if_right", ascending=False, inplace=True)
    df.drop_duplicates(subset=["enzyme", "substrate", "Binding"], inplace=True)
    return dfold, dfnew, df

In [281]:
with open(f"../output/puepp_{topk}.csv", "w") as f:
    for queryname in querynamelist:
        dfold, dfnew, df = pueppcombine(queryname)
        sum_right_num, sum_all_num = pueppcount(df)
        f.write(f"{queryname},{sum_right_num}, {sum_all_num}\n")
        df.to_csv(f"../out4cj/puepp_{queryname}_{topk}.csv", index=False)

filtered_rows 30
sum_right_num 1
sum_all_num 30
sumsum_num 5405 

